In [ ]:
from typing import Dict

import lightgbm as lgb
import polars as pl
import shap
from zh import CLOSE, HIGH, LOW, OPEN, OPEN_TIME, SYMBOL, UTC, scan

data = scan("5m").lazy()

In [ ]:
hold = 2
start_date = pl.datetime(2020, 1, 1, time_zone=UTC)
train_date = pl.datetime(2024, 1, 1, time_zone=UTC)
val_date = pl.datetime(2025, 1, 1, time_zone=UTC)
test_date = pl.datetime(2025, 12, 31, time_zone=UTC)
lag = 3

In [ ]:
# kline features
def kline_features() -> Dict[str, pl.Expr]:
    upper_shadow = (HIGH - pl.max_horizontal(OPEN, CLOSE)) / OPEN
    lower_shadow = (pl.min_horizontal(OPEN, CLOSE) - LOW) / OPEN
    amp = (HIGH - LOW) / OPEN
    inline_ret = CLOSE / OPEN - 1
    features = {
        "upper_shadow": upper_shadow,
        "lower_shadow": lower_shadow,
        "amp": amp,
        "inline_ret": inline_ret,
    }
    return features


def add_features(data: pl.LazyFrame, lag: int = lag) -> pl.LazyFrame:
    return data.with_columns(
        [
            feat.shift(i).over(SYMBOL).alias(f"{name}_{i}")
            for name, feat in kline_features().items()
            for i in range(lag)
        ]
    ).drop_nulls()


data = add_features(data.sort([SYMBOL, OPEN_TIME]))

features = kline_features()


In [ ]:
train_data = (
    data.filter((OPEN_TIME >= start_date) & (OPEN_TIME < train_date))
    .with_columns(
        (((CLOSE.shift(1 - hold) / OPEN - 1) / hold).shift(-1).over(SYMBOL)).alias(
            "target"
        )
    )
    .drop_nulls()
)
train_X = train_data.select(
    [f"{name}_{i}" for name in features.keys() for i in range(lag)]
)
train_y = train_data.select("target")
train_X, train_y = pl.collect_all([train_X, train_y], engine="streaming")
train_X = train_X.to_numpy()
train_y = train_y.to_series().to_numpy()
dtrain = lgb.Dataset(
    train_X,
    label=train_y,
    feature_name=[f"{name}_{i}" for name in features.keys() for i in range(lag)],
)

In [ ]:
lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "force_row_wise": True,
    "seed": 42,
}
bst = lgb.train(lgb_params, dtrain, num_boost_round=100)

In [ ]:
train_X = (
    pl.DataFrame(train_X, schema=dtrain.feature_name).sample(100000, seed=42).to_numpy()
)

In [ ]:
explainer = shap.TreeExplainer(bst)
shap_values = explainer.shap_values(train_X)
shap.summary_plot(
    shap_values,
    train_X,
    feature_names=[f"{name}_{i}" for name in features.keys() for i in range(lag)],
)